In [ ]:
from pathlib import Path
import pandas as pd 

df = pd.read_csv("../../data/interim/extract/extract_all.csv")
df.shape

(21003, 26)

In [2]:
df.head()

,mention,parcours,promotion,situation,n_interroges,taux_reponse,n_en_emploi,n_recherche_emploi,n_en_etudes,n_autre_situation,...,missions,categorie_emploi,type_emploi,secteur_employeur,activite_employeur,lieu_emploi,autre_diplome,temps_partiel,revenu_net,emploi_stable
0,Arts,Cinéma et nouvelles images,2019,1er décembre 2021,9,78.0,5,1,1,0,...,NaN,Employé ou ouvrier,Non stable,Public,Enseignement,Hauts-de-France hors MEL,Aucun,True,NaN,False
1,Arts,Cinéma et nouvelles images,2019,1er décembre 2021,9,78.0,5,1,1,0,...,Processus logistique,Cadre,Stable,Privé,NaN,Île-de-France,Aucun,True,NaN,True
2,Arts,Cinéma et nouvelles images,2019,1er décembre 2021,9,78.0,5,1,1,0,...,NaN,Cadre,Non stable,Public,Enseignement,Hauts-de-France hors MEL,Aucun,False,1808.0,False
3,Arts,Cinéma et nouvelles images,2019,1er décembre 2021,9,78.0,5,1,1,0,...,NaN,Cadre,Stable,Public,Enseignement,Hauts-de-France hors MEL,Aucun,False,2000.0,True
4,Arts,Cinéma et nouvelles images,2019,1er décembre 2021,9,78.0,5,1,1,0,...,NaN,Cadre,Stable,Public,Enseignement,Île-de-France,Aucun,True,NaN,True


In [3]:
df.columns

Index(['mention', 'parcours', 'promotion', 'situation', 'n_interroges',
       'taux_reponse', 'n_en_emploi', 'n_recherche_emploi', 'n_en_etudes',
       'n_autre_situation', 'n_total_repondants', 'n_emploi_stable',
       'n_emploi_non_stable', 'effectif_insuffisant', 'diplome',
       'intitule_emploi', 'missions', 'categorie_emploi', 'type_emploi',
       'secteur_employeur', 'activite_employeur', 'lieu_emploi',
       'autre_diplome', 'temps_partiel', 'revenu_net', 'emploi_stable'],
      dtype='str')

# FOCALISATION SUR LA POPULATION DE REFERENCE : 

Nous cherchons ici à modéliser la probabilité qu'un diplômé exerçant un emploi à temps plein perçoive un salaire supérieur à la médiane de référence. 
En effet nous ne pouvons ici pas traiter les emplois à temps partiel car l'ODIF ne publie que les salaires pour les emplois à temps plein et précise lorsque l'absence de salaire déclaré s'explique par le fait que l'individu sondé exerce un temploi à temps partiel. 

Voir la documentation officielle des variables de l'enquête : "revenu net en euros : il s'agit du revenu net mensuel, primes comprises, en euros (ou indication si l’emploi est à temps partiel)". 

Nous ne disposons donc même pas d'information qui permettrait de convertir les salaires à temps partiel en leurs équivalents temps plein. 
Nous devons donc restreindre notre base de données aux individus occupants un emploi à temps plein :

In [4]:
print("taille initiale : ", df.shape)

df = df[df["temps_partiel"] == 0]

print("taille après traitment : ", df.shape)

taille initiale :  (21003, 26)
taille après traitment :  (19863, 26)


# 1 VALEURS MANQUANTES :

In [5]:
df.isna().sum()

mention                 1730
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            1
missions                3743
categorie_emploi           2
type_emploi               10
secteur_employeur         52
activite_employeur       160
lieu_emploi              190
autre_diplome              5
temps_partiel              0
revenu_net              2244
emploi_stable             10
dtype: int64

# 1.1 Salaires 

De même, comme notre objectif est la modélisation d'un phénomène relatif au salaire, nous ne pouvons conserver dans la base des observations pour lesquels le salaire n'est pas renseigné. 
Etant donné qu'il s'agit de notre variable cible, nous préférons ne faire aucune imputation aux valeurs manquantes ici, car cela pourrait très (trop) fortement biaiser le modèle. 

Néanmoins par soucis méthodologique, vérifions que la suppression de ce sous-ensemble de la base (salaire non déclaré) n'entraîne pas de biais majeur, notamment en comparant les deux populations : 

In [6]:
df["revenu_manquant"] = df["revenu_net"].isna()

df_avec_salaire = df[df["revenu_manquant"] == False]
df_sans_salaire = df[df["revenu_manquant"] == True]

print("effectifs :")
print("salaire renseigné :", len(df_avec_salaire))
print("salaire manquant :", len(df_sans_salaire))

print("\nProportion de salaires manquants :")
print(f"{df['revenu_manquant'].mean():.2%}")

effectifs :
salaire renseigné : 17619
salaire manquant : 2244

Proportion de salaires manquants :
11.30%


In [7]:
from scipy.stats import mannwhitneyu, chi2_contingency
from sklearn.preprocessing import LabelEncoder
import numpy as np 

def cramers_v(table):
    chi2 = chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    r, k = table.shape

    if n == 0 or min(r-1, k-1) == 0:
        return np.nan

    return np.sqrt(chi2 / (n * min(r-1, k-1)))


resultats = []

variables = [
    col for col in df.columns
    if col not in ["revenu_net", "revenu_manquant"]
]

for col in variables:

    temp = df[[col, "revenu_manquant"]].dropna()

    if temp[col].nunique() <= 1:
        continue

    # Variables quantitatives
    if pd.api.types.is_numeric_dtype(df[col]):

        x1 = temp.loc[temp["revenu_manquant"] == False, col]
        x2 = temp.loc[temp["revenu_manquant"] == True, col]

        if len(x1) < 2 or len(x2) < 2:
            continue

        stat, pvalue = mannwhitneyu(
            x1, x2, alternative="two-sided"
        )

        pooled_std = np.sqrt(
            (x1.var() + x2.var()) / 2
        )

        smd = (
            (x1.mean() - x2.mean()) / pooled_std
            if pooled_std > 0 else np.nan
        )

        resultats.append({
            "Variable": col,
            "Type": "Quantitative",
            "Valeur_avec_salaire": x1.mean(),
            "Valeur_sans_salaire": x2.mean(),
            "Taille_effet": abs(smd),
            "p_value": pvalue
        })

    # Variables qualitatives
    else:

        table = pd.crosstab(
            temp[col], temp["revenu_manquant"]
        )

        if min(table.shape) < 2:
            continue

        chi2, pvalue, _, _ = chi2_contingency(table)

        resultats.append({
            "Variable": col,
            "Type": "Qualitative",
            "Valeur_avec_salaire": np.nan,
            "Valeur_sans_salaire": np.nan,
            "Taille_effet": cramers_v(table),
            "p_value": pvalue
        })


resultats = pd.DataFrame(resultats)

resultats["p_value_ajustee"] = (
    resultats["p_value"] * len(resultats)
).clip(upper=1)

resultats = resultats.sort_values(
    "Taille_effet", ascending=False
)

display(resultats)

,Variable,Type,Valeur_avec_salaire,Valeur_sans_salaire,Taille_effet,p_value,p_value_ajustee
15,missions,Qualitative,NaN,NaN,0.977205,2.858464e-01,1.000000e+00
14,intitule_emploi,Qualitative,NaN,NaN,0.708281,4.479240e-02,1.000000e+00
1,parcours,Qualitative,NaN,NaN,0.209960,4.318755e-17,9.933136e-16
2,promotion,Quantitative,2020.109711,2020.631462,0.198463,1.158881e-17,2.665426e-16
0,mention,Qualitative,NaN,NaN,0.128169,1.426900e-15,3.281871e-14
20,lieu_emploi,Qualitative,NaN,NaN,0.090010,7.983926e-32,1.836303e-30
19,activite_employeur,Qualitative,NaN,NaN,0.081328,1.653705e-13,3.803521e-12
3,situation,Qualitative,NaN,NaN,0.065832,1.968641e-16,4.527873e-15
8,n_en_etudes,Quantitative,0.976786,1.074866,0.060693,1.161773e-03,2.672078e-02
21,autre_diplome,Qualitative,NaN,NaN,0.054706,7.686433e-05,1.767880e-03


Ces résultats d'analyse d'associtions entre nos deux groupes sont très encourageants car ils montrent que dans l'ensemble les deux populations ne sont que relativement peu différentes. 
Seul fait marquant : l'association reste très forte pour le descriptif des missions, le titre de l'emploi et le master (mention et parcours). Cela signifie que certains masters mènent à des professions qui sont plus souvent occupés à temps partiel. 
C'est un résultat logique et prévisible, qui n'introduit pas de biais dans la réponse à notre problématique (on se concentre sur la modélisation du salaire pour les emplois à TEMPS PLEIN) mais qui réduira notre nombre d'observations pour ces formations cependant. 

Nous procédons ainsi à la supressions des individus aux salaires non renseignés :

In [8]:
print("avant traitement : ", df.shape)
df = df[df["revenu_manquant"]==False]
print("apres traitement : ", df.shape)


avant traitement :  (19863, 27)
apres traitement :  (17619, 27)


## 1.2 Mention du diplôme :

In [9]:
verif = df[df["mention"].isna()]
verif.head()

,mention,parcours,promotion,situation,n_interroges,taux_reponse,n_en_emploi,n_recherche_emploi,n_en_etudes,n_autre_situation,...,categorie_emploi,type_emploi,secteur_employeur,activite_employeur,lieu_emploi,autre_diplome,temps_partiel,revenu_net,emploi_stable,revenu_manquant
13785,NaN,Administration territoriale (Parcours transver...,2016,1er décembre 2018,10,80.0,7,0,1,0,...,Prof. interm.,Non stable,Privé,Transport et entreposage,Métropole Européenne de Lille,Autre Master,False,1750.0,False,False
13786,NaN,Administration territoriale (Parcours transver...,2016,1er décembre 2018,10,80.0,7,0,1,0,...,Cadre,Non stable,Public,Administration publique,Métropole Européenne de Lille,Aucun,False,1900.0,False,False
13787,NaN,Administration territoriale (Parcours transver...,2016,1er décembre 2018,10,80.0,7,0,1,0,...,Employé ou ouvrier,Stable,Public,Administration publique,Métropole Européenne de Lille,Aucun,False,1842.0,True,False
13788,NaN,Administration territoriale (Parcours transver...,2016,1er décembre 2018,10,80.0,7,0,1,0,...,Prof. interm.,Stable,Privé,Commerce,Métropole Européenne de Lille,Aucun,False,1350.0,True,False
13789,NaN,Administration territoriale (Parcours transver...,2016,1er décembre 2018,10,80.0,7,0,1,0,...,Cadre,Stable,Public,Administration publique,Reste de la France,Aucun,False,2042.0,True,False


En 2016 on a juste le nom des parcours, ils avaient pas encore tt regroupés par mentions. 
Cependant les masters existent généralement sur une longue période sans changer de nom. On peut donc raisonablement supposer que si en 2017 , 2018 , 2019 etc. un parcours est rattaché à la mention X, lui attribuer cette même mention pour 2016 est tout à fait cohérent. 
Par exemple pour SIAD en 2016 les 4 parcours n'ont aucune mentions, or nous savons (car nous sommes étudiants dans ce master), que ces 4 parcours appartiennent au même master. 

Pour faciliter ce travail avec les centaines de master qui existent à l'université, un LLM a comparé les mentions et les parcours des autres années avec 2016 et trouve que dans 68% des cas, il y a un nom de mention unique pour l'ensemble des années suivantes pour ce parcours. Nous réalisons donc cette imputation : 

In [10]:
def cle_texte(texte):
    """normalisation des libellés.
    2 noms  qui ne diffèrent que par la casse, les accents ou les apostrophes ont la même clé."""
    texte = str(texte).translate(str.maketrans({"’": "'", "‘": "'", "–": "-", "—": "-"}))  
    texte = unicodedata.normalize("NFKD", texte).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", texte.lower()).strip()

In [11]:
import unicodedata
import re
# Référence : toutes les lignes dont la mention est connue (2017 et après)
connues = df.loc[df["mention"].notna(), ["parcours", "mention"]].copy()
connues["cle_parcours"] = connues["parcours"].map(cle_texte)
connues["cle_mention"] = connues["mention"].map(cle_texte)

# Nombre de mentions distinctes associées à chaque nom de parcours
nb_mentions = connues.groupby("cle_parcours")["cle_mention"].nunique()

# On ne garde que les parcours qui n'ont qu'une seule mention possible
parcours_uniques = nb_mentions[nb_mentions == 1].index

# Écriture de la mention à reprendre : la plus fréquente dans les données
mention_du_parcours = (
    connues[connues["cle_parcours"].isin(parcours_uniques)]
    .groupby("cle_parcours")["mention"]
    .agg(lambda s: s.value_counts().idxmax())
)

print(f"{len(parcours_uniques)} parcours avec une seule mention possible (sur {len(nb_mentions)} parcours connus)")

401 parcours avec une seule mention possible (sur 429 parcours connus)


In [12]:
mention_avant = df["mention"].copy()

# Lignes sans mention, et mention trouvée grâce au nom du parcours (vide si ambigu ou introuvable)
sans_mention = df["mention"].isna()
cle_parcours = df.loc[sans_mention, "parcours"].map(cle_texte)
mention_trouvee = cle_parcours.map(mention_du_parcours).dropna()

# Remplissage + repère des valeurs déduites
df["mention_inferee"] = False
df.loc[mention_trouvee.index, "mention"] = mention_trouvee
df.loc[mention_trouvee.index, "mention_inferee"] = True

print(f"{len(mention_trouvee)} ligne(s) complétée(s) sur {sans_mention.sum()} sans mention")

1170 ligne(s) complétée(s) sur 1588 sans mention


In [13]:
# 1) Aucune mention existante ne doit avoir changé
deja_connues = mention_avant.notna()
print("Mentions existantes inchangées :", (df.loc[deja_connues, "mention"] == mention_avant[deja_connues]).all())

# 2) Ce qui reste sans mention : plusieurs mentions possibles, ou parcours introuvable
restantes = df[df["mention"].isna()].copy()
restantes["cause"] = restantes["parcours"].map(cle_texte).map(
    lambda k: "plusieurs mentions possibles" if k in nb_mentions.index else "parcours introuvable"
)
print("\nLignes restantes sans mention :")
print(restantes.groupby("promotion")["cause"].value_counts().to_string())

Mentions existantes inchangées : True

Lignes restantes sans mention :
promotion  cause                       
2016       parcours introuvable            269
           plusieurs mentions possibles    149


In [14]:
verif = df[df["mention"].isna()]
verif["parcours"].unique()

<ArrowStringArray>
[                                  'Administration territoriale (Parcours transversal)',
                                              'Affaires et négociation internationales',
                                        'Art et responsabilité sociale - International',
                                  'Communication et management des ressources humaines',
                                                     'Communication interne et externe',
                        'Communication interne et externe - formation en apprentissage',
                        'Didactique des langues - français langue étrangère et seconde',
                                                        'Éthique, politique et société',
                                                      'Gestion des sites du patrimoine',
                                                   'Management de projets touristiques',
                                                              'Métiers de la rédaction',
  

A voir ce qu'on fait pour ces 400 individus. 

In [15]:
df.isna().sum()

mention                  418
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            0
missions                2948
categorie_emploi           0
type_emploi                6
secteur_employeur         10
activite_employeur        76
lieu_emploi               70
autre_diplome              1
temps_partiel              0
revenu_net                 0
emploi_stable              6
revenu_manquant            0
mention_inferee            0
dtype: int64

# 1.3 SECTEUR EMPLOYEUR

Pour 10 individus le secteur d'activité n'est pas specifié. 
Nous allons les analyser en détails pour chercher à savoir si il est possible de déterminer ce dernier, notamment depuis le descriptif des missions etc. 

In [16]:
secteur_manquant = df[df["secteur_employeur"].isna()]

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(secteur_manquant[['missions', 'intitule_emploi', "mention"]])

,missions,intitule_emploi,mention
1243,Traitement de documents,Employé-e administratif-ve,Finance
1501,NaN,Emploi intermédiaire,Management et administration des entreprises
2027,Recherche et enseignement,Doctorant-e,Chimie physique et analytique
10574,NaN,Aide physicien·ne médical·e,Physique fondamentale et applications
13562,"Gestion d’un portefeuille de clients, des flux, relation clientèle, amélioration de ces flux, amélioration du cycle logistique",Responsable en amélioration des flux logistiques,Management et administration des entreprises
17828,Recherche,Chercheur-se doctorant-e,"Sciences de la Terre et des planètes, environnement"
19506,Achats,Assistant-e de gestion,Management
19652,"Réponse aux appels d'offres, analyse pricing, communication",Business developer (volontariat international),Management et commerce international
20440,Mise en place et gestion d'études cliniques,Attaché-e de recherche clinique,Ingénierie de la santé
20480,Recherche et enseignement,Doctorant-e,Ingénierie de la santé


In [17]:
# 1243 : Employé-e administratif-ve
# Impossible à déterminer : ce poste existe dans les secteurs public, privé et associatif. La ligne sera supprimée.
df.loc[1243, "secteur_employeur"] = None

# 1501 : Emploi intermédiaire
# Impossible à déterminer : intitulé trop générique.
df.loc[1501, "secteur_employeur"] = None

# 2027 : Doctorant-e
# Public car la recherche et l'enseignement sont fréquemment et principalement exercés dans les universités et organismes publics.
df.loc[2027, "secteur_employeur"] = "Public"

# 10574 : Aide physicien·ne médical·e
# Impossible a determiner il peut travailler en publique cpmme en pv
df.loc[10574, "secteur_employeur"] = None

# 13562 : Responsable en amélioration des flux logistiques
# Privé car fonction typique d'une entreprise commerciale ou industrielle, bcp plus rare dans l'associatif ou le publique, en plus dans un master management des ENTREPRISES
df.loc[13562, "secteur_employeur"] = "Privé"

# 17828 : Chercheur-se doctorant-e
# assigne public pareil souvent exercé dans une université etc
df.loc[17828, "secteur_employeur"] = "Public"

# 19506 : Assistant-e de gestion
# peut pas determiner, poste transversal présent dans les trois secteurs.
df.loc[19506, "secteur_employeur"] = None

# 19652 : Business developer (volontariat international)
# Privé car fonction commerciale généralement exercée en entreprise en plus après un master de commercer et le VIE s'effectue en entreprise
df.loc[19652, "secteur_employeur"] = "Privé"

# 20440 : Attaché-e de recherche clinique
# impossible de deviner
df.loc[20440, "secteur_employeur"] = None

# 20480 : Doctorant-e
# Public encore une fois pr les memes raisons
df.loc[20480, "secteur_employeur"] = "Public"


# Suppression des individus pour lesquels le secteur est indéterminable
df = df.dropna(subset=["secteur_employeur"])

Pour valider nos hypothèses on peut également regarder les secteurs dans lesquels travaillent majoritairement les individus avec le même intitulé de poste :

In [18]:

imputations = {
    "Doctorant-e": "Public",
    "Aide physicien·ne médical·e": "Public",
    "Responsable en amélioration des flux logistiques": "Privé",
    "Chercheur-se doctorant-e": "Public",
    "Business developer (volontariat international)": "Privé"
}

resultats = []

for intitule, secteur_impute in imputations.items():

    # Sélection des individus ayant le même intitulé
    subset = df[
        (df["intitule_emploi"] == intitule) &
        (df["secteur_employeur"].notna())
    ]

    # Effectif total avec secteur connu
    total = len(subset)

    if total == 0:
        print(f"\n{intitule} : aucun secteur renseigné dans le df.")
        continue

    # Répartition des secteurs en pourcentage
    proportions = (
        subset["secteur_employeur"]
        .value_counts(normalize=True)
        .mul(100)
    )

    # Résultats
    for secteur in ["Public", "Privé", "Associatif"]:
        resultats.append({
            "Intitulé emploi": intitule,
            "Secteur imputé": secteur_impute,
            "Secteur observé": secteur,
            "Effectif": (subset["secteur_employeur"] == secteur).sum(),
            "Proportion (%)": round(proportions.get(secteur, 0), 2),
            "Effectif total": total
        })

resultats_df = pd.DataFrame(resultats)

display(resultats_df)


Aide physicien·ne médical·e : aucun secteur renseigné dans le df.


,Intitulé emploi,Secteur imputé,Secteur observé,Effectif,Proportion (%),Effectif total
0,Doctorant-e,Public,Public,124,89.86,138
1,Doctorant-e,Public,Privé,10,7.25,138
2,Doctorant-e,Public,Associatif,4,2.90,138
3,Responsable en amélioration des flux logistiques,Privé,Public,0,0.00,1
4,Responsable en amélioration des flux logistiques,Privé,Privé,1,100.00,1
5,Responsable en amélioration des flux logistiques,Privé,Associatif,0,0.00,1
6,Chercheur-se doctorant-e,Public,Public,148,86.55,171
7,Chercheur-se doctorant-e,Public,Privé,20,11.70,171
8,Chercheur-se doctorant-e,Public,Associatif,3,1.75,171
9,Business developer (volontariat international),Privé,Public,0,0.00,1


Nos stats confirment que les docteurs sont surements dans le publique. 
Le VIE confirme que le business developper est très certainement dans le privé.
Il y a que pour l'amélioration des flux logisitique ou un doute peut persister, mais son master se concentre sur le management des entreprises, ils sont principalement dans le pv :

In [19]:
subset = df[df["mention"] == "Management et administration des entreprises"]
display((subset["secteur_employeur"].value_counts(normalize=True, dropna=False) * 100).round(2))

secteur_employeur
Privé         74.62
Public        17.31
Associatif     8.08
Name: proportion, dtype: float64

In [20]:
df.isna().sum()

mention                  418
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            0
missions                2946
categorie_emploi           0
type_emploi                6
secteur_employeur          0
activite_employeur        74
lieu_emploi               68
autre_diplome              1
temps_partiel              0
revenu_net                 0
emploi_stable              6
revenu_manquant            0
mention_inferee            0
dtype: int64

## 1.4 type d'emploi 

Même logique pour éviter de supprimer inutilement des lignes, voyons s'il est possible d'imputer ces résultats

In [21]:
type_manquant = df[df["type_emploi"].isna()]

with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(type_manquant[['missions', 'intitule_emploi', "mention"]])

,missions,intitule_emploi,mention
13499,NaN,Chargé·e de mission relation entreprise,Management de l’innovation
16060,Traduction de séries pour le doublage,Adaptateur-trice audiovisuel-le,"Traduction, interprétation"
16063,"Traduction et adaptation de l'anglais vers le français de programmes pour la télé, le cinéma et le jeu vidéo",Traducteur-trice adaptateur-trice,"Traduction, interprétation"
17524,NaN,Conducteur-trice de bus,Chimie
19489,NaN,Responsable de résidence,Management
20039,Recherche et enseignement,Doctorant-e,Biotechnologies


In [22]:
df["type_emploi"].unique()

<ArrowStringArray>
['Non stable', 'Stable', nan]
Length: 3, dtype: str

In [23]:
indices = [13499, 16060, 16063, 17524, 19489, 20039]
cibles = df.loc[indices, ["missions", "intitule_emploi", "mention"]]
df_connu = df[df["emploi_stable"].notna()].copy()


print("stats par metiers")

noms_emplois = cibles["intitule_emploi"].dropna().unique()

for emploi in noms_emplois:
    subset = df_connu[df_connu["intitule_emploi"] == emploi]
    print(emploi)
    print("Effectif total :", len(subset))
    print("Effectifs par stabilité :")
    display(subset["emploi_stable"].value_counts(dropna=False))
    print("Répartition en pourcentage :")
    display(
        (subset["emploi_stable"].value_counts(normalize=True) * 100)
        .round(2)
    )


stats par metiers
Chargé·e de mission relation entreprise
Effectif total : 0
Effectifs par stabilité :


Series([], Name: count, dtype: int64)

Répartition en pourcentage :


Series([], Name: proportion, dtype: float64)

Adaptateur-trice audiovisuel-le
Effectif total : 2
Effectifs par stabilité :


emploi_stable
True    2
Name: count, dtype: int64

... (14 outputs left out)

Doctorant-e
Effectif total : 137
Effectifs par stabilité :


emploi_stable
False    136
True       1
Name: count, dtype: int64

Répartition en pourcentage :


emploi_stable
False    99.27
True      0.73
Name: proportion, dtype: float64

In [24]:
print("stat par mentions")
mentions = cibles["mention"].dropna().unique()

for mention in mentions:
    subset = df_connu[df_connu["mention"] == mention]
    print("Effectif total :", len(subset))
    print(mention)
    print("Effectifs par stabilité :")
    display(subset["emploi_stable"].value_counts(dropna=False))
    print("Répartition en pourcentage :")
    display(
        (subset["emploi_stable"].value_counts(normalize=True) * 100)
        .round(2)
    )

stat par mentions
Effectif total : 83
Management de l’innovation
Effectifs par stabilité :


emploi_stable
True     61
False    22
Name: count, dtype: int64

Répartition en pourcentage :


emploi_stable
True     73.49
False    26.51
Name: proportion, dtype: float64

Effectif total : 71
Traduction, interprétation
Effectifs par stabilité :


emploi_stable
True     68
False     3
Name: count, dtype: int64

... (10 outputs left out)

Effectif total : 127
Biotechnologies
Effectifs par stabilité :


emploi_stable
True     69
False    58
Name: count, dtype: int64

Répartition en pourcentage :


emploi_stable
True     54.33
False    45.67
Name: proportion, dtype: float64

In [25]:
# Imputation de la variable emploi_stable
# True = Stable ; False = Non stable

# 13499 : Chargé·e de mission relation entreprise
# Aucun individu comparable par métier. Mention Management de l'innovation : 73,49 % stables. pas de pbs a dire que c stable normalement
df.loc[13499, "type_emploi"] = "Stable"

# 16060 : Adaptateur-trice audiovisuel-le
# 2 observations comparables : 100 % stables. Mention Traduction, interprétation : 95,77 % stables.
df.loc[16060, "type_emploi"] = "Stable"

# 16063 : Traducteur-trice adaptateur-trice
# Stable avec 3 observations comparables : 100 % stables. Mention Traduction, interprétation : 95,77 % stables.
df.loc[16063, "type_emploi"] = "Stable"

# 17524 : Conducteur-trice de bus
# 1 observation comparable : non stable. La mention est pas liée au métier, on ne peut rien savoir = sup
df.loc[17524, "type_emploi"] = None

# 19489 : Responsable de résidence
# Meme chose que pour l'indiv précédent
df.loc[19489, "type_emploi"] = None

# 20039 : Doctorant-e
# 137 observations comparables : 99,27 % non stables.
df.loc[20039, "type_emploi"] = "Non stable"

df = df.dropna(subset=["type_emploi"])

In [26]:
df.isna().sum()

mention                  418
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            0
missions                2944
categorie_emploi           0
type_emploi                0
secteur_employeur          0
activite_employeur        74
lieu_emploi               68
autre_diplome              1
temps_partiel              0
revenu_net                 0
emploi_stable              4
revenu_manquant            0
mention_inferee            0
dtype: int64

# 1.5 : Activité de l'employeur 

Même chose que les sections précédentes 

In [27]:
act_manquante = df[df["activite_employeur"].isna()]
df["activite_employeur"].unique()

<ArrowStringArray>
[                                               'Enseignement',
                   'Arts, spectacles et activités récréatives',
                                     'Administration publique',
                                'Autres activités de services',
 'Activités informatiques, traitement, hébergement de données',
                                                    'Commerce',
                                    'Transport et entreposage',
                             'Santé humaine et action sociale',
                                 'Hébergement et restauration',
     'Ingénierie recherche et développement études techniques',
          'Activités de services administratifs et de soutien',
                                'Édition presse communication',
                                                   'Industrie',
                        'Activités financières et d'assurance',
                               'Publicité et études de marché',
                  'Co

In [28]:
act_manquante.index.unique()

Index([  305,   558,   578,   667,   713,   738,   841,   962,  1141,  1363,
        1442,  1464,  1658,  1795,  2080,  2229,  2258,  2842,  7378,  8723,
        9403, 10182, 10724, 11621, 11859, 12522, 12870, 12951, 13424, 13562,
       15437, 16345, 16365, 16415, 16501, 17274, 17368, 17670, 17722, 17780,
       17853, 17921, 18147, 18268, 18326, 18417, 18436, 18478, 18490, 18499,
       18527, 18632, 18729, 18730, 18777, 18834, 18982, 18983, 19075, 19098,
       19143, 19167, 19185, 19257, 19415, 19631, 19659, 19673, 19887, 19960,
       20147, 20480, 20642, 20659],
      dtype='int64')

In [29]:
indices = [305,   558,   578,   667,   713,   738,   841,   962,  1141,  1363,
        1442,  1464,  1658,  1795,  2080,  2229,  2258,  2842,  7378,  8723,
        9403, 10182, 10724, 11621, 11859, 12522, 12870, 12951, 13424, 13562,
       15437, 16345, 16365, 16415, 16501, 17274, 17368, 17670, 17722, 17780,
       17853, 17921, 18147, 18268, 18326, 18417, 18436, 18478, 18490, 18499,
       18527, 18632, 18729, 18730, 18777, 18834, 18982, 18983, 19075, 19098,
       19143, 19167, 19185, 19257, 19415, 19631, 19659, 19673, 19887, 19960,
       20147, 20480, 20642, 20659]  

cibles = df.loc[indices,["missions", "intitule_emploi", "mention"]]
df_connu = df[df["activite_employeur"].notna()].copy()

for emploi in cibles["intitule_emploi"].dropna().unique():

    subset = df_connu[df_connu["intitule_emploi"] == emploi]
    print(emploi)
    print("Effectif total :", len(subset))

    display(
        subset["activite_employeur"]
        .value_counts()
        .to_frame("Effectif")
    )

    display(
        (subset["activite_employeur"].value_counts(normalize=True) * 100)
        .round(2)
        .to_frame("Pourcentage")
    )

Vendeur-se
Effectif total : 11


,Effectif
activite_employeur,
Commerce,9
"Agriculture,sylviculture et pêche",1
Industrie,1


,Pourcentage
activite_employeur,
Commerce,81.82
"Agriculture,sylviculture et pêche",9.09
Industrie,9.09


Adaptateur-trice de doublage
Effectif total : 0


,Effectif
activite_employeur,


,Pourcentage
activite_employeur,


... (185 outputs left out)

,Pourcentage
activite_employeur,
"Comptabilité, gestion, conseil en management",100.0


Consultant-e ingénieur-e dispositifs médicaux
Effectif total : 0


,Effectif
activite_employeur,


,Pourcentage
activite_employeur,


In [30]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(df.loc[indices, ["intitule_emploi", "mention"]])

,intitule_emploi,mention
305,Vendeur-se,Lettres
558,Adaptateur-trice de doublage,"Traduction, interprétation"
578,Traducteur-trice,"Traduction, interprétation"
667,Assistant-e ressources humaines,Administration économique et sociale (AES)
713,Inspecteur-trice des finances publiques,Administration publique
738,Collaborateur-trice comptable,"Comptabilité, contrôle, audit"
841,Gestionnaire back office,Contrôle de gestion et audit organisationnel
962,Notaire stagiaire,Droit notarial
1141,Juriste droit social,Droit social
1363,Consultant-e déploiement,Management


In [31]:
# Imputations de activite_employeur à partir des répartitions observées par métier
# On impute des classes majoritaires quand c'est assez clair. Au moindre doute on supprime la ligne

imputations = {

    # Vendeur-se : Commerce = 9/11 (81,82 %)
    305: "Commerce",

    # Traducteur-trice : Autres activités scientifiques et techniques = 15/20 (75 %)
    578: "Autres activités scientifiques et techniques",

    # Inspecteur-trice des finances publiques : Administration publique = 25/25 (100 %)
    713: "Administration publique",

    # Collaborateur-trice comptable : Comptabilité, gestion, conseil en management = 33/36 (91,67 %)
    738: "Comptabilité gestion conseil en management",

    # Notaire stagiaire : Activités juridiques = 44/48 (91,67 %)
    962: "Activités juridiques",

    # Consultant-e décisionnel : Activités informatiques = 16/17 (94,12 %)
    1464: "Activités informatiques, traitement, hébergement de données",

    # Chargé-e de clientèle particuliers : Activités financières et d'assurance = 4/4 (100 %)
    1795: "Activités financières et d'assurance",

    # Ingénieur-e d'application : Commerce = 10/14 (71,43 %)
    10724: "Commerce",

    # Chercheur-se doctorant-e : Enseignement = 113/168 (67,26 %)
    17722: "Enseignement",

    # Chercheur-se doctorant-e : Enseignement = 113/168 (67,26 %)
    17780: "Enseignement",

    # Chercheur-se doctorant-e : Enseignement = 113/168 (67,26 %)
    17853: "Enseignement",

    # Attaché-e de recherche clinique : Santé humaine et action sociale = 23/37 (62,16 %)
    17921: "Santé humaine et action sociale",

    # Psychologue spécialisé-e en neuropsychologie : Santé humaine et action sociale = 14/15 (93,33 %)
    18417: "Santé humaine et action sociale",

    # Psychologue : Santé humaine et action sociale = 259/361 (71,75 %)
    18436: "Santé humaine et action sociale",

    # Psychologue : Santé humaine et action sociale = 259/361 (71,75 %)
    18478: "Santé humaine et action sociale",

    # Psychologue : Santé humaine et action sociale = 259/361 (71,75 %)
    18490: "Santé humaine et action sociale",

    # Psychologue : Santé humaine et action sociale = 259/361 (71,75 %)
    18499: "Santé humaine et action sociale",

    # Psychologue : Santé humaine et action sociale = 259/361 (71,75 %)
    18527: "Santé humaine et action sociale",

    # Avocat-e : Activités juridiques = 18/19 (94,74 %)
    18834: "Activités juridiques",

    # Consultant-e en recrutement : Services administratifs et de soutien = 11/17 (64,71 %)
    18983: "Activités de services administratifs et de soutien",

    # Avocat-e : Activités juridiques = 18/19 (94,74 %)
    19098: "Activités juridiques",

    # Avocat-e : Activités juridiques = 18/19 (94,74 %)
    19167: "Activités juridiques",

    # Consultant-e en recrutement : Services administratifs et de soutien = 11/17 (64,71 %)
    19659: "Activités de services administratifs et de soutien",

    # Chercheur-se doctorant-e : Enseignement = 113/168 (67,26 %)
    20147: "Enseignement",

    # Chercheur-se doctorant-e : Enseignement = 113/168 (67,26 %)
    20480: "Enseignement",
}


# Individus pour lesquels l'activité employeur est indéterminable
# Justifications :
# - effectif nul : aucune répartition disponible 
# - effectif inférieur à 10 : données trop peu nombreuses 
# - absence d'activité atteignant 50 % 
# - égalité entre plusieurs activités majoritaires.

indices_none = [
    558,   # Adaptateur-trice de doublage : aucun individu observé (n=0)
    667,   # Gestionnaire back office : deux catégories à égalité (1/2 chacune)
    841,   # Consultant-e déploiement : aucun individu observé (n=0)
    1141,  # Réceptionniste : effectif trop faible (n=2), malgré 100 % dans une catégorie
    1363,  # Micro entrepreneur-e : aucun individu observé (n=0)
    1442,  # Programmeur-se informatique : aucun individu observé (n=0)
    1658,  # Responsable qualité : activité majoritaire = 20/41 (48,78 %), sous le seuil
    2080,  # Ingénieur-e d'affaires et business manager : n=0
    2229,  # Cadre ingénieur-e : n=0
    2258,  # Chargé-e de recouvrement : n=0
    2842,  # Contrôleur-euse de gestion : activité majoritaire = 14/46 (30,43 %)
    7378,  # Ingénieur-e en automatique : n=0
    8723,  # Ingénieur-e en modélisation et simulation énergétique : n=0
    9403,  # Chargé-e d'affaires réglementaires internationales : n=0
    10182, # Adjoint-e des opérations financières : n=0
    11621, # Juriste : activité majoritaire = 29/127 (22,83 %)
    11859, # Assistant-e de communication : n=0
    12522, # Responsable en amélioration des flux logistiques : n=0
    12870, # Coordinateur-trice projet affaires réglementaires : n=0
    12951, # Coach de professionnels : n=0
    13424, # Chargé-e de projet européen et international : n=0
    13562, # Analyste système : n=0
    15437, # Bibliothécaire : égalité entre Administration publique et Enseignement (1/2)
    16345, # Psychologue : à vérifier selon l'intitulé exact de l'individu
    16415, # Formateur-trice digital learning : n=0
    16501, # Traducteur-trice spécialisé-e : n=0
    17274, # Juriste marchés publics : égalité entre deux catégories (5/12 chacune)
    17368, # Chargé-e de développement : égalité entre deux catégories (3/12 chacune)
    17670, # Juriste en droit pénal : effectif trop faible (n=1)
    18268, # Chargé-e des relations institutionnelles : n=0
    18326, # Commercial-e : activité majoritaire = 5/14 (35,71 %)
    18632, # Psychologue clinicien-ne : activité majoritaire = 61/68 (89,71 %) ; à vérifier
    18729, # Traducteur-trice technique : effectif trop faible (n=1)
    18730, # Chargé-e d'études commerciales : effectif trop faible (n=1)
    18777, # Chargé-e de recrutement : activité majoritaire = 11/41 (26,83 %)
    18982, # Auto-entrepreneur-e : activité majoritaire = 4/12 (33,33 %)
    19075, # Juriste marchés publics : égalité entre deux catégories (5/12 chacune)
    19143, # Chargé-e de développement : égalité entre deux catégories (3/12 chacune)
    19185, # Gestionnaire ressources humaines : activité majoritaire = 5/11 (45,45 %)
    19257, # Consultant-e : activité majoritaire = 14/36 (38,89 %)
    19415, # Commercial-e : activité majoritaire = 5/14 (35,71 %)
    19631, # Social media manager : activité majoritaire = 3/9 (33,33 %)
    19673, # Ingénieur-e de recherche : à examiner selon l'intitulé exact
    19887, # Doctorant-e : à examiner selon l'intitulé exact
    19960, # Regulatory affairs consultant : effectif trop faible (n=1)
    20642, # Consultant-e ingénieur-e dispositifs médicaux : n=0
    20659, # À vérifier selon l'intitulé exact
    16365, # pas assez dobs
    18147 # pareil
]

# Application des imputations
df.loc[list(imputations.keys()), "activite_employeur"] = pd.Series(imputations)

# Affectation des valeurs indéterminables
df.loc[indices_none, "activite_employeur"] = None

# Suppression des lignes dont l'activité employeur reste manquante
df = df.dropna(subset=["activite_employeur"])


Nb ici pour gagner du temps sur l'analyse nous avons utilisé un LLM en lui fournissant notre tableau d'analyse des répartitions par métier et la liste des indexs a imputer avec le métier correspondant. 
Nous avons ensuite définie les règles suivantes : on impute la classe majoritaire pour ce métier si elle est cosntatée à 60% ou plus sur un jeu d'au moins 10 observations. Dans le cas contraire on supprime l'index correspondant

In [33]:
df.isna().sum()

mention                  418
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            0
missions                2923
categorie_emploi           0
type_emploi                0
secteur_employeur          0
activite_employeur         0
lieu_emploi               62
autre_diplome              1
temps_partiel              0
revenu_net                 0
emploi_stable              4
revenu_manquant            0
mention_inferee            0
dtype: int64

# 1.6 Lieu de l'emploi 

Ici contrairement aux sections précédentes, il semble difficile d'imputer une classe majoritaire sans biaiser les choses. 
Si on peut aisément supposer que certains masters et/ou métiers augmentent très fortement les chances qu'un individu pour lequel on ne connait pas le type d'employeur, le secteur d'activité ou le niveau de stabilité de l'emploi appartienne à une classe donnée (par exemple le master forme spécifiquement à tel métier, ou tel métier est spécifique à tel secteur d'activité) sans introduire de biais notable; Le choix du lieu de travail par les individus dépend trop de la situation personnel de ces derniers et par conséquent nous préférons donc simplement supprimer les individus pour lesquels nous ne disposons pas de ces informations plutot que de réaliser une imputation trop aléatoire

In [34]:
print(df.shape)
df = df.dropna(subset=["lieu_emploi"])
print(df.shape)


(17563, 28)
(17501, 28)


In [36]:
df.isna().sum()

mention                  416
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            0
missions                2893
categorie_emploi           0
type_emploi                0
secteur_employeur          0
activite_employeur         0
lieu_emploi                0
autre_diplome              1
temps_partiel              0
revenu_net                 0
emploi_stable              4
revenu_manquant            0
mention_inferee            0
dtype: int64

# 1.7 Autre diplôme 

In [37]:
df["autre_diplome"].value_counts(normalize=True) * 100

autre_diplome
Aucun                                                      85.062857
Autre Master                                                5.120000
Autre diplôme (bac+5)                                       2.280000
Diplôme école ingénieur (bac+5)                             2.177143
Autre diplôme (bac+6)                                       1.417143
Autre master                                                0.931429
Diplôme école commerce gestion (bac+5)                      0.800000
Diplôme d’école d’ingénieurs (bac+5)                        0.422857
Diplôme d’école paramédicale ou sociale (bac+5)             0.308571
Sc Po (bac+5)                                               0.268571
Mastère (bac+6)                                             0.251429
Mastère (bac+5)                                             0.222857
Doctorat                                                    0.211429
Diplôme spécifique aux études médicales (bac+6 ou plus)     0.120000
Diplôme d’une autre 

Dans 85% des cas l'individu sondé n'a aucun autre diplôme, le risque d'introduire un biais en imputant la classe majoritaire dans 85% de la base à un seul individu étant proche de 0 nous faisons ici ce choix : 

In [39]:
diplome_manquant = df[df["autre_diplome"].isna()]
diplome_manquant.index.unique()


Index([15688], dtype='int64')

In [41]:
df.loc[15688, "autre_diplome"] = "Aucun"
df.isna().sum()

mention                  416
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            0
missions                2893
categorie_emploi           0
type_emploi                0
secteur_employeur          0
activite_employeur         0
lieu_emploi                0
autre_diplome              0
temps_partiel              0
revenu_net                 0
emploi_stable              4
revenu_manquant            0
mention_inferee            0
dtype: int64

## 1.8 Stabilité de l'emploi 

In [45]:
emploi_manquant = df[df["emploi_stable"].isna()]
emploi_manquant[["intitule_emploi", "mention"]]

,intitule_emploi,mention
13499,Chargé·e de mission relation entreprise,Management de l’innovation
16060,Adaptateur-trice audiovisuel-le,"Traduction, interprétation"
16063,Traducteur-trice adaptateur-trice,"Traduction, interprétation"
20039,Doctorant-e,Biotechnologies


In [47]:
individus = [(13499, "Chargé·e de mission relation entreprise", "Management de l’innovation"),
    (16060, "Adaptateur-trice audiovisuel-le", "Traduction, interprétation"),
    (16063, "Traducteur-trice adaptateur-trice", "Traduction, interprétation"),
    (20039, "Doctorant-e", "Biotechnologies")]


for index, metier, mention in individus:

    print(f"INDIVIDU {index} : {metier} | {mention}")

    print("répartition de emploi_stable pour le mm métier")
    repartition_metier = (
        df.loc[df["intitule_emploi"] == metier, "emploi_stable"]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    print("Nombre d'individus :", df.loc[df["intitule_emploi"] == metier].shape[0])
    print(repartition_metier)

    # 2. Répartition selon la même mention de master
    print(" rpartition de emploi_stable pour la mm mention")

    repartition_mention = (
        df.loc[df["mention"] == mention, "emploi_stable"]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    print("Nombre d'individus :", df.loc[df["mention"] == mention].shape[0])
    print(repartition_mention)

INDIVIDU 13499 : Chargé·e de mission relation entreprise | Management de l’innovation
répartition de emploi_stable pour le mm métier
Nombre d'individus : 1
emploi_stable
NaN    100.0
Name: proportion, dtype: float64
 rpartition de emploi_stable pour la mm mention
Nombre d'individus : 84
emploi_stable
True     72.62
False    26.19
NaN       1.19
Name: proportion, dtype: float64
INDIVIDU 16060 : Adaptateur-trice audiovisuel-le | Traduction, interprétation
répartition de emploi_stable pour le mm métier
Nombre d'individus : 3
emploi_stable
True    66.67
NaN     33.33
Name: proportion, dtype: float64
 rpartition de emploi_stable pour la mm mention
Nombre d'individus : 69
emploi_stable
True     92.75
False     4.35
NaN       2.90
Name: proportion, dtype: float64
INDIVIDU 16063 : Traducteur-trice adaptateur-trice | Traduction, interprétation
répartition de emploi_stable pour le mm métier
Nombre d'individus : 4
emploi_stable
True    75.0
NaN     25.0
Name: proportion, dtype: float64
 rpartitio

Il n'y a que pour le doctorant qu'un nb suffisant d'individu partageants le même métier existe pour pouvoir donner du crédit à ces répartitions. 
Les 3 autres individus sont supprimés : 

In [48]:
print(df.shape)
df.loc[20039, "emploi_stable"] = True
df = df.dropna(subset=["emploi_stable"])
print(df.shape)


(17501, 28)
(17498, 28)


In [50]:
df.isna().sum()

mention                  416
parcours                   0
promotion                  0
situation                  0
n_interroges               0
taux_reponse               0
n_en_emploi                0
n_recherche_emploi         0
n_en_etudes                0
n_autre_situation          0
n_total_repondants         0
n_emploi_stable            0
n_emploi_non_stable        0
effectif_insuffisant       0
diplome                    0
intitule_emploi            0
missions                2892
categorie_emploi           0
type_emploi                0
secteur_employeur          0
activite_employeur         0
lieu_emploi                0
autre_diplome              0
temps_partiel              0
revenu_net                 0
emploi_stable              0
revenu_manquant            0
mention_inferee            0
dtype: int64

il restera à décider ce qu'on fait pour les 416 mentions manquantes. Pour les missions manquantes, si on les utilise ce sera à travers la représentation (probablement diminué) d'un espace vetoriel donc ce n'est pas gênant de garder des valeurs manquantes. On pourra traiter ces cas au moment de la construction de l'espace et de sa projection (si on retient cette méthode)

# Fin du traitement des valeurs manquantes 

Le traitement des valeurs manquantes nous fait passer de ~21 000 observations à ~ 17 500 ce qui reste une bonne base pour un modèle sur de la donnée tabulaire. 